# Agente de Triagem Médica com Rede Bayesiana
### Tratamento das bases de dados e criação da rede Bayesiana

**Disciplina:** Introdução à Inteligência Artificial  
**Objetivo:** Preparar os dados, construir a Rede Bayesiana e salvar o modelo treinado.



---
## 1 - Importação das Bibliotecas

- **pandas / numpy** — manipulação e análise dos dados
- **pgmpy** — modelagem e treinamento da Rede Bayesiana
- **pickle** — serialização do modelo treinado
- **matplotlib** — visualizações


In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

print("✅ Bibliotecas carregadas com sucesso!")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")


---
## 2 — Análise exploratória dos Dados

In [ ]:
df = pd.read_csv("../datasets/dataset.csv")

print(f"Total de doenças únicas: {df['Disease'].nunique()}")
print()
df.head()

In [ ]:
contagem = df['Disease'].str.strip().value_counts()
print("Registros por doença:")
print(contagem.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
contagem.plot(kind='barh', ax=ax, color='#028090', edgecolor='none')
ax.set_title('Registros por Doença no Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Quantidade de registros')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../datasets/distribuicao_doencas.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Gráfico salvo!")


In [ ]:
colunas_sintoma = [c for c in df.columns if c.startswith('Symptom')]
todos_sintomas = pd.Series(df[colunas_sintoma].values.ravel()).dropna().str.strip().unique()

print(f"Total de colunas de sintoma: {len(colunas_sintoma)}")
print(f"Total de sintomas únicos: {len(todos_sintomas)}")
print()
print("Valores nulos por coluna de sintoma:")
print(df[colunas_sintoma].isnull().sum().to_string())


---
## 3 — Limpeza e Cálculo das Probabilidades

1. **Remoção de espaços** nos nomes de doenças e sintomas
2. **Cálculo de P(sintoma | doença)** — para cada par (doença, sintoma), calculamos a frequência relativa com que aquele sintoma aparece nas linhas daquela doença

In [ ]:
df['Disease'] = df['Disease'].str.strip()
for col in colunas_sintoma:
    df[col] = df[col].str.strip()

resultados = []

for doenca in df['Disease'].unique():
    df_doenca = df[df['Disease'] == doenca]
    total_linhas = len(df_doenca)

    sintomas_doenca = pd.Series(
        df_doenca[colunas_sintoma].values.ravel()
    ).dropna().str.strip()

    contagem = sintomas_doenca.value_counts()

    for sintoma, count in contagem.items():
        resultados.append({
            'Disease': doenca,
            'Sintoma': sintoma,
            'contagem': count,
            'P_sintoma_dado_doenca': count / total_linhas
        })

probabilidades = pd.DataFrame(resultados)
probabilidades.to_csv('../datasets/probabilidades.csv', index=False)

print()
print(probabilidades[probabilidades['Disease'] == 'Dengue'][
    ['Sintoma', 'P_sintoma_dado_doenca']
].sort_values('P_sintoma_dado_doenca', ascending=False).to_string(index=False))


---
## 4 — Base para tradução dos dados


In [ ]:
traducao_doencas = {
    "Fungal infection": "Infecção fúngica",
    "Allergy": "Alergia",
    "GERD": "Refluxo gastroesofágico",
    "Chronic cholestasis": "Colestase crônica",
    "Drug Reaction": "Reação a medicamento",
    "Peptic ulcer diseae": "Úlcera péptica",
    "AIDS": "AIDS",
    "Diabetes": "Diabetes",
    "Gastroenteritis": "Gastroenterite",
    "Bronchial Asthma": "Asma brônquica",
    "Hypertension": "Hipertensão",
    "Migraine": "Enxaqueca",
    "Cervical spondylosis": "Espondilose cervical",
    "Paralysis (brain hemorrhage)": "Paralisia (hemorragia cerebral)",
    "Jaundice": "Icterícia",
    "Malaria": "Malária",
    "Chicken pox": "Catapora",
    "Dengue": "Dengue",
    "Typhoid": "Febre tifoide",
    "hepatitis A": "Hepatite A",
    "Hepatitis B": "Hepatite B",
    "Hepatitis C": "Hepatite C",
    "Hepatitis D": "Hepatite D",
    "Hepatitis E": "Hepatite E",
    "Alcoholic hepatitis": "Hepatite alcoólica",
    "Tuberculosis": "Tuberculose",
    "Common Cold": "Resfriado comum",
    "Pneumonia": "Pneumonia",
    "Dimorphic hemmorhoids(piles)": "Hemorroidas",
    "Heart attack": "Infarto",
    "Varicose veins": "Varizes",
    "Hypothyroidism": "Hipotireoidismo",
    "Hyperthyroidism": "Hipertireoidismo",
    "Hypoglycemia": "Hipoglicemia",
    "Osteoarthristis": "Osteoartrite",
    "Arthritis": "Artrite",
    "(vertigo) Paroymsal  Positional Vertigo": "Vertigem posicional",
    "Acne": "Acne",
    "Urinary tract infection": "Infecção urinária",
    "Psoriasis": "Psoríase",
    "Impetigo": "Impetigo",
}

traducao_sintomas = {
    "itching": "coceira", "skin_rash": "erupção_cutânea",
    "nodal_skin_eruptions": "erupções_nodulares", "continuous_sneezing": "espirros_contínuos",
    "shivering": "tremores", "chills": "calafrios", "joint_pain": "dor_nas_articulações",
    "stomach_pain": "dor_no_estômago", "acidity": "acidez", "ulcers_on_tongue": "úlceras_na_língua",
    "muscle_wasting": "perda_muscular", "vomiting": "vômito", "burning_micturition": "ardência_ao_urinar",
    "spotting_ urination": "manchas_na_urina", "fatigue": "fadiga", "weight_gain": "ganho_de_peso",
    "anxiety": "ansiedade", "cold_hands_and_feets": "mãos_e_pés_frios", "mood_swings": "mudanças_de_humor",
    "weight_loss": "perda_de_peso", "restlessness": "agitação", "lethargy": "letargia",
    "patches_in_throat": "manchas_na_garganta", "irregular_sugar_level": "nível_irregular_de_açúcar",
    "cough": "tosse", "high_fever": "febre_alta", "sunken_eyes": "olhos_fundos",
    "breathlessness": "falta_de_ar", "sweating": "suor_excessivo", "dehydration": "desidratação",
    "indigestion": "indigestão", "headache": "dor_de_cabeça", "yellowish_skin": "pele_amarelada",
    "dark_urine": "urina_escura", "nausea": "náusea", "loss_of_appetite": "perda_de_apetite",
    "pain_behind_the_eyes": "dor_atrás_dos_olhos", "back_pain": "dor_nas_costas",
    "constipation": "constipação", "abdominal_pain": "dor_abdominal", "diarrhoea": "diarreia",
    "mild_fever": "febre_leve", "yellow_urine": "urina_amarela", "yellowing_of_eyes": "amarelamento_dos_olhos",
    "acute_liver_failure": "falência_hepática_aguda", "fluid_overload": "sobrecarga_de_fluidos",
    "swelling_of_stomach": "inchaço_do_estômago", "swelled_lymph_nodes": "linfonodos_inchados",
    "malaise": "mal_estar", "blurred_and_distorted_vision": "visão_turva", "phlegm": "catarro",
    "throat_irritation": "irritação_na_garganta", "redness_of_eyes": "vermelhidão_nos_olhos",
    "sinus_pressure": "pressão_nos_seios_nasais", "runny_nose": "coriza", "congestion": "congestão",
    "chest_pain": "dor_no_peito", "weakness_in_limbs": "fraqueza_nos_membros",
    "fast_heart_rate": "frequência_cardíaca_elevada", "pain_during_bowel_movements": "dor_ao_evacuar",
    "pain_in_anal_region": "dor_na_região_anal", "bloody_stool": "fezes_com_sangue",
    "irritation_in_anus": "irritação_no_ânus", "neck_pain": "dor_no_pescoço", "dizziness": "tontura",
    "cramps": "cãibras", "bruising": "hematomas", "obesity": "obesidade", "swollen_legs": "pernas_inchadas",
    "swollen_blood_vessels": "vasos_sanguíneos_inchados", "puffy_face_and_eyes": "rosto_e_olhos_inchados",
    "enlarged_thyroid": "tireoide_aumentada", "brittle_nails": "unhas_quebradiças",
    "swollen_extremeties": "extremidades_inchadas", "excessive_hunger": "fome_excessiva",
    "extra_marital_contacts": "contatos_extraconjugais", "drying_and_tingling_lips": "lábios_ressecados_e_formigando",
    "slurred_speech": "fala_arrastada", "knee_pain": "dor_no_joelho",
    "hip_joint_pain": "dor_na_articulação_do_quadril", "muscle_weakness": "fraqueza_muscular",
    "stiff_neck": "rigidez_no_pescoço", "swelling_joints": "articulações_inchadas",
    "movement_stiffness": "rigidez_no_movimento", "spinning_movements": "sensação_de_rotação",
    "loss_of_balance": "perda_de_equilíbrio", "unsteadiness": "instabilidade",
    "weakness_of_one_body_side": "fraqueza_em_um_lado_do_corpo", "loss_of_smell": "perda_do_olfato",
    "bladder_discomfort": "desconforto_na_bexiga", "foul_smell_of urine": "urina_com_cheiro_forte",
    "continuous_feel_of_urine": "vontade_contínua_de_urinar", "passage_of_gases": "eliminação_de_gases",
    "internal_itching": "coceira_interna", "toxic_look_(typhos)": "aparência_tóxica",
    "depression": "depressão", "irritability": "irritabilidade", "muscle_pain": "dor_muscular",
    "altered_sensorium": "alteração_sensorial", "red_spots_over_body": "manchas_vermelhas_no_corpo",
    "belly_pain": "dor_na_barriga", "abnormal_menstruation": "menstruação_irregular",
    "dischromic _patches": "manchas_discrômicas", "watering_from_eyes": "lacrimejamento",
    "increased_appetite": "apetite_aumentado", "polyuria": "poliúria", "family_history": "histórico_familiar",
    "mucoid_sputum": "escarro_mucoide", "rusty_sputum": "escarro_enferrujado",
    "lack_of_concentration": "falta_de_concentração", "visual_disturbances": "distúrbios_visuais",
    "receiving_blood_transfusion": "recebeu_transfusão_de_sangue",
    "receiving_unsterile_injections": "recebeu_injeções_não_esterilizadas", "coma": "coma",
    "stomach_bleeding": "sangramento_estomacal", "distention_of_abdomen": "distensão_abdominal",
    "history_of_alcohol_consumption": "histórico_de_consumo_de_álcool",
    "fluid_overload.1": "sobrecarga_de_fluidos_2", "blood_in_sputum": "sangue_no_escarro",
    "prominent_veins_on_calf": "veias_proeminentes_na_panturrilha", "palpitations": "palpitações",
    "painful_walking": "dificuldade_ao_caminhar", "pus_filled_pimples": "espinhas_com_pus",
    "blackheads": "cravos", "scurring": "descamação", "skin_peeling": "descascamento_da_pele",
    "silver_like_dusting": "descamação_prateada", "small_dents_in_nails": "pequenas_depressões_nas_unhas",
    "inflammatory_nails": "unhas_inflamadas", "blister": "bolhas",
    "red_sore_around_nose": "ferida_vermelha_ao_redor_do_nariz", "yellow_crust_ooze": "crostas_amarelas",
}

In [ ]:
df_pt = df.copy()
df_pt['Disease'] = df_pt['Disease'].map(traducao_doencas)
for col in colunas_sintoma:
    df_pt[col] = df_pt[col].map(traducao_sintomas)

df_pt.to_csv('../datasets/dataset_pt.csv', index=False)

prob_pt = probabilidades.copy()
prob_pt['Disease'] = prob_pt['Disease'].map(traducao_doencas)
prob_pt['Sintoma'] = prob_pt['Sintoma'].map(traducao_sintomas)
prob_pt.to_csv('../datasets/probabilidades_pt.csv', index=False)

print(f"Dataset traduzido: {df_pt.shape}")
print(f"Probabilidades traduzidas: {prob_pt.shape}")
print()

---
## 5 — Construção da Rede Bayesiana

### Estrutura adotada: Naive Bayes

```
Disease ──→ sintoma_1
Disease ──→ sintoma_2
Disease ──→ sintoma_3
    ...
Disease ──→ sintoma_131
```

O nó `Disease` é o nó pai (a causa) e cada sintoma é um nó filho (o efeito). Esta arquitetura é chamada de **Naive Bayes** — assume que os sintomas são condicionalmente independentes dado o diagnóstico, o que é uma simplificação, mas altamente eficaz na prática.

### Aprendizado dos parâmetros

Utilizamos **Maximum Likelihood Estimation (MLE)** para aprender as tabelas de probabilidade condicional (CPTs) diretamente do dataset. Para cada par (doença, sintoma), o MLE calcula:

$$P(sintoma = 1 \mid doença) = \frac{\text{linhas com o sintoma}}{\text{total de linhas da doença}}$$


In [ ]:
df_bin = pd.read_csv('../datasets/dataset_pt.csv')
colunas_sintoma_pt = [c for c in df_bin.columns if c.startswith('Symptom')]

todos_sintomas_pt = pd.Series(
    df_bin[colunas_sintoma_pt].values.ravel()
).dropna().unique()

colunas_binarias = {}
colunas_binarias['Disease'] = df_bin['Disease']
for sintoma in todos_sintomas_pt:
    colunas_binarias[sintoma] = df_bin[colunas_sintoma_pt].apply(
        lambda row: 1 if sintoma in row.values else 0, axis=1
    )

df_binario = pd.DataFrame(colunas_binarias)
print(f"Dataset binário: {df_binario.shape}")
print(f"   {df_binario.shape[0]} pacientes × {df_binario.shape[1]-1} sintomas + 1 coluna de doença")
df_binario.head(3)


In [ ]:
arestas = [('Disease', sintoma) for sintoma in todos_sintomas_pt]
model = BayesianNetwork(arestas)

print(f"   Nós: {len(model.nodes())}")
print(f"   Arestas: {len(model.edges())}")


In [ ]:
print("Aprendendo parâmetros do dataset...")
model.fit(df_binario, estimator=MaximumLikelihoodEstimator)

valido = model.check_model()
print(f"✅ Modelo válido: {valido}")
print(f"   CPTs aprendidas para {len(model.cpds)} variáveis")


In [ ]:
cpd_disease = model.get_cpds('Disease')
print("Estados possíveis de Disease:")
for estado in cpd_disease.state_names['Disease']:
    print(f"  - {estado}")

print()
print(f"Prior uniforme: P(Disease = qualquer doença) = {1/41:.4f} ≈ {100/41:.1f}%")

cpd_febre = model.get_cpds('febre_alta')
estados = cpd_disease.state_names['Disease'][:5]
indices = [list(cpd_disease.state_names['Disease']).index(e) for e in estados]

print("\nP(febre_alta | Disease) para 5 doenças:")
for i, doenca in zip(indices, estados):
    p = cpd_febre.values[0][i]
    print(f"  {doenca:<40} {p:.2f}")

---
## 6 — Salvar o Modelo


In [ ]:
with open('../datasets/modelo_rede.pkl', 'wb') as f:
    pickle.dump({
        'model': model,
        'sintomas': list(todos_sintomas_pt)
    }, f)

print("Modelo salvo em datasets/modelo_rede.pkl")
print()

with open('../datasets/modelo_rede.pkl', 'rb') as f:
    verificacao = pickle.load(f)

m = verificacao['model']
s = verificacao['sintomas']
print(f"Verificação do arquivo salvo:")
print(f"  Nós no modelo: {len(m.nodes())}")
print(f"  Sintomas salvos: {len(s)}")
print(f"  Modelo válido: {m.check_model()}")
